# Importar Bibliotecas Necesarias
Importar pandas, numpy, sklearn (Pipeline, RandomForestClassifier, OneHotEncoder, SimpleImputer, train_test_split, classification_report, accuracy_score), joblib y las funciones personalizadas necesarias de los módulos utils.

In [ ]:
import pandas as pd
import numpy as np
from statistics import mean
import os
import sys
import joblib # For saving the model
import json

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Ruta absoluta al directorio src
src_path = os.path.abspath("../src") # Adjust if the notebook location changes relative to src
if src_path not in sys.path:
    sys.path.append(src_path)

# Import custom utility functions
from utils.exploration_utils import (
    read_csv_files,
    preprocess_data,
    generate_target_variable,
    evaluar_escenarios_trading,
    sma_engineering_from_close,
    feature_engineering_rsi,
    bollinger_bands_engineering,
    macd_engineering,
    atr_engineering,
    adx_engineering,
    stochastic_engineering,
    parabolic_sar_engineering,
    round_number_pivot_engineering,
    calcular_features_vela,
    create_candle_features_vector,
    create_basic_patterns,
    create_pattern_mapping,
    apply_pattern_mapping
)
from utils.support_resistance import calculate_progressive_support_resistance, calcular_metricas_fila
from utils.viz_utils import plot_candlestick_with_clusters_plotly, analizar_metricas_por_umbral
from sklearn.cluster import KMeans # For clustering steps if included in pipeline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100) # Adjust as needed

# Reload modules if needed during development
import importlib
import utils.exploration_utils
import utils.support_resistance
import utils.viz_utils
importlib.reload(utils.exploration_utils)
importlib.reload(utils.support_resistance)
importlib.reload(utils.viz_utils)

<module 'utils.viz_utils' from 'c:\\Users\\Administrador\\Documents\\Proyectos\\Trading\\ForexDataExtractorProject\\src\\utils\\viz_utils.py'>

# Cargar Datos Iniciales
Cargar el conjunto de datos principal (por ejemplo, desde un archivo parquet '../data/parquet/EURUSD/EURUSD_5mins.parquet').

In [ ]:
# In the cell with id 'f253e1f3' replace the existing source code with this:
# Read Data from CSV files:
folder_path = '../data/EURUSD/5mins' # Define the folder containing CSV files
print(f"Attempting to load data from CSV files in: {folder_path}")

try:
    # Check if the directory exists
    if not os.path.isdir(folder_path):
         raise FileNotFoundError(f"Error: Directory not found at {folder_path}. Please check the path.")

    df_EURUSD_5min = read_csv_files(folder_path) # Use the utility function to read CSVs

    # Check if data was loaded
    if df_EURUSD_5min.empty:
        print("Warning: No data loaded. The DataFrame is empty. Check the CSV files or the read_csv_files function.")
    else:
        print(f"Data loaded successfully from CSVs in {folder_path}")
        print(f"Initial DataFrame shape: {df_EURUSD_5min.shape}")

        # Apply initial preprocessing (e.g., column renaming, basic cleaning)
        print("Applying initial preprocessing (preprocess_data)...")
        df_EURUSD_5min = preprocess_data(df_EURUSD_5min)
        print(f"DataFrame shape after preprocess_data: {df_EURUSD_5min.shape}")

        # --- Explicit Datetime Preprocessing (similar to realtime_processor goal) ---
        print("Applying explicit datetime preprocessing...")
        if 'datetime' not in df_EURUSD_5min.columns:
            # Attempt to find common alternatives if 'datetime' is missing after preprocess_data
            if 'Date' in df_EURUSD_5min.columns:
                 print("Found 'Date' column, renaming to 'datetime'.")
                 df_EURUSD_5min.rename(columns={'Date': 'datetime'}, inplace=True)
            elif 'time' in df_EURUSD_5min.columns:
                 print("Found 'time' column, renaming to 'datetime'.")
                 df_EURUSD_5min.rename(columns={'time': 'datetime'}, inplace=True)
            else:
                 raise KeyError("Critical Error: 'datetime' column not found after preprocessing and no alternative ('Date', 'time') found.")

        # Convert to datetime objects, coercing errors
        # Try common formats or let pandas infer; adjust format string if needed
        print(f"Converting '{'datetime'}' column to datetime objects...")
        df_EURUSD_5min['datetime'] = pd.to_datetime(df_EURUSD_5min['datetime'], errors='coerce')

        # Check for conversion errors (NaT values)
        nat_count = df_EURUSD_5min['datetime'].isna().sum()
        if nat_count > 0:
            print(f"Warning: {nat_count} rows had datetime conversion errors and are now NaT.")
            # Optional: Drop rows with NaT datetimes if they cannot be used
            # df_EURUSD_5min.dropna(subset=['datetime'], inplace=True)
            # print(f"Dropped {nat_count} rows with NaT datetimes.")

        # Ensure timezone is UTC
        if df_EURUSD_5min['datetime'].dt.tz is None:
            print("Localizing datetime column to UTC...")
            df_EURUSD_5min['datetime'] = df_EURUSD_5min['datetime'].dt.tz_localize('UTC')
        else:
            print("Converting datetime column to UTC...")
            df_EURUSD_5min['datetime'] = df_EURUSD_5min['datetime'].dt.tz_convert('UTC')

        # Verify final dtype
        if not pd.api.types.is_datetime64_any_dtype(df_EURUSD_5min['datetime']):
             print("Warning: 'datetime' column is STILL not in a recognized datetime format after explicit conversion.")
        else:
             print("'datetime' column successfully processed to timezone-aware datetime objects (UTC).")

        # Ensure sorting by datetime after all processing
        print("Sorting DataFrame by datetime...")
        df_EURUSD_5min = df_EURUSD_5min.sort_values('datetime').reset_index(drop=True)
        print("DataFrame sorted.")
        # --- End Explicit Datetime Preprocessing ---

except FileNotFoundError as fnf_error:
     print(fnf_error)
     raise # Re-raise the exception to stop the notebook if the directory is critical

except KeyError as ke:
     print(ke) # Print the specific KeyError message
     raise # Re-raise to indicate failure

except Exception as e:
    print(f"An unexpected error occurred during data loading or preprocessing: {e}")
    import traceback
    traceback.print_exc() # Print full traceback for unexpected errors
    raise # Re-raise to indicate failure

# Display head if data loaded successfully
if 'df_EURUSD_5min' in locals() and not df_EURUSD_5min.empty:
    print("\nDisplaying first few rows of the loaded and preprocessed data:")
    df_EURUSD_5min.head()
else:
    print("\nCould not display head as DataFrame is not available or empty.")


# Definir Funciones de Ingeniería de Características y Preprocesamiento

Aquí definimos las funciones que encapsulan los pasos de ingeniería de características y preprocesamiento. Estas funciones serán utilizadas dentro del Pipeline de Scikit-Learn, a menudo a través de `FunctionTransformer`.

1.  **Calcular Niveles S/R:** Usa `calculate_progressive_support_resistance`.
2.  **Calcular Métricas de Niveles:** Usa `calcular_metricas_fila`.
3.  **Generar Target:** Usa `evaluar_escenarios_trading` y crea la variable objetivo multiclase.
4.  **Ingeniería de Características Técnicas:** Aplica SMA, RSI, BBands, MACD, ATR, ADX, Stochastic, SAR, Round Numbers, etc.
5.  **Características de Velas:** Calcula `calcular_features_vela` y `create_candle_features_vector`.
6.  **(Opcional) Clustering y Patrones:** Incluye clustering (`KMeans`) y mapeo de patrones (`create_basic_patterns`, etc.) si se desea.

*Nota: Para simplificar, muchas de estas funciones ya existen en los módulos `utils`. Las envolveremos o llamaremos directamente en los transformadores del pipeline.*

In [ ]:
# --- Wrapper Functions for Pipeline ---

# 1. & 2. Support/Resistance and Metrics
# Note: calculate_progressive_support_resistance is computationally intensive and stateful.
# It might be better to run it once offline and save the results, then load them.
# If run in pipeline, it needs careful handling (e.g., custom transformer).
# For this example, we assume S/R levels are pre-calculated or handled differently.
# We will focus on applying metrics based on pre-existing S/R data if available.

# Placeholder function if S/R levels need calculation within pipeline (complex)
# def calculate_sr_and_metrics_transformer(df):
#     # ... implementation using calculate_progressive_support_resistance ...
#     # ... then apply calcular_metricas_fila ...
#     # This is complex due to the progressive nature. Pre-calculation is recommended.
#     print("Applying S/R calculation and metrics (Placeholder)")
#     # Assuming 'relevant_levels' column exists or is calculated here
#     if 'relevant_levels' in df.columns:
#          metricas = df.apply(calcular_metricas_fila, axis=1)
#          df = pd.concat([df, metricas], axis=1)
#     return df




# 4. Technical Indicators Feature Engineering
def apply_technical_indicators(df):
    """Applies various technical indicators."""
    print("Applying technical indicators (SMA, RSI, BBands, MACD, ATR, ADX, Stoch, SAR)...")
    df = sma_engineering_from_close(df, [10, 40, 100])
    df = feature_engineering_rsi(df)
    df = bollinger_bands_engineering(df, periods=[20, 50], std_devs=[2.0, 2.5])
    df = macd_engineering(df, fast_periods=[12, 8], slow_periods=[26, 21], signal_periods=[9], lookback=5)
    df = atr_engineering(df, periods=[14, 21], lookback=5)
    df = adx_engineering(df, periods=[14, 21], lookback=5)
    df = stochastic_engineering(df, k_periods=[14, 21], d_periods=[3, 5], lookback=5, slow=True)
    df = parabolic_sar_engineering(df)
    df = round_number_pivot_engineering(df)
    return df

# 5. Candle Features Engineering
def apply_candle_features(df, n_candles=6):
    """Calculates basic candle features and vectorizes them."""
    print(f"Applying candle features (vectorizing last {n_candles} candles)...")
    df = calcular_features_vela(df)
    features_to_vectorize = ['body_size', 'upper_wick', 'lower_wick', 'total_range',
                             'body_ratio', 'upper_wick_ratio', 'lower_wick_ratio']
    # Ensure columns exist before vectorizing
    features_to_vectorize = [f for f in features_to_vectorize if f in df.columns]
    if features_to_vectorize:
        df = create_candle_features_vector(df, feature_columns=features_to_vectorize, n_candles=n_candles)
    else:
        print("Warning: No candle features found to vectorize.")
    return df

# 6. (Optional) Clustering and Pattern Features
# These often require fitting on training data only. Use custom transformers for this.
# Example structure for a custom transformer:
from sklearn.base import BaseEstimator, TransformerMixin

class CandleClusterer(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=3, n_candles=5, random_state=42):
        self.n_clusters = n_clusters
        self.n_candles = n_candles # For pattern creation later
        self.random_state = random_state
        self.kmeans_ = None
        self.imputer_ = None
        self.features_actuales = ['body_size', 'upper_wick', 'lower_wick', 'total_range',
                                  'body_ratio', 'upper_wick_ratio', 'lower_wick_ratio']

    def fit(self, X, y=None):
        print(f"Fitting CandleClusterer (k={self.n_clusters})...")
        # Ensure candle features are calculated first if not present
        if not all(f in X.columns for f in self.features_actuales):
             # This indicates an issue with pipeline order or missing step
             raise ValueError("Candle features must be calculated before clustering.")

        X_features = X[self.features_actuales].copy()
        # Impute missing values before clustering
        self.imputer_ = SimpleImputer(strategy='mean')
        X_imputed = self.imputer_.fit_transform(X_features)

        self.kmeans_ = KMeans(n_clusters=self.n_clusters, random_state=self.random_state, n_init=10)
        self.kmeans_.fit(X_imputed)
        return self

    def transform(self, X):
        print("Transforming with CandleClusterer...")
        if self.kmeans_ is None or self.imputer_ is None:
            raise RuntimeError("CandleClusterer must be fitted before transforming.")

        X_transformed = X.copy()
        X_features = X_transformed[self.features_actuales].copy()
        X_imputed = self.imputer_.transform(X_features) # Use fitted imputer

        clusters = self.kmeans_.predict(X_imputed)
        X_transformed['candle_cluster'] = clusters.astype(str) # Store as string/category

        # Optionally create patterns based on these clusters
        # Note: Pattern mapping should also ideally be fit on training data
        # This part is complex for a simple FunctionTransformer, better in another custom transformer or offline
        # X_transformed['pattern'] = create_basic_patterns(X_transformed, cluster_column='candle_cluster', n_candles=self.n_candles)
        # print("Candle clusters and basic patterns added.")
        print("Candle clusters added.")
        return X_transformed

# --- Helper to select feature columns ---
def select_features(df, feature_list):
    """Selects specified features, handling missing ones."""
    print(f"Selecting features...")
    available_features = [f for f in feature_list if f in df.columns]
    missing = set(feature_list) - set(available_features)
    if missing:
        print(f"Warning: Missing expected features: {missing}")
    return df[available_features]

# --- Define Feature List ---
# Start with a base list and add features generated by the functions
# This list needs careful management as features are added/removed
# It's often better to define this *after* seeing the output of the feature engineering steps
# For now, let's define a placeholder based on the original notebook's RF features
# We will refine this after defining the pipeline steps

base_numerical_features = [ # Examples, adjust based on actual generated features
    'open', 'high', 'low', 'close', # Keep basic OHLC? Maybe not for prediction.
    'volume',
    # Add features generated by apply_technical_indicators
    'SMA_10', 'SMA_40', 'SMA_100', 'diff_SMA_10_SMA_40', 'ratio_SMA_10_SMA_40',
    'RSI_14', 'RSI_7', 'RSI_21', 'dist_rsi_70', 'dist_rsi_30',
    'BB_20_20_middle', 'BB_20_20_upper', 'BB_20_20_lower', 'BB_20_20_percent_b', 'BB_20_20_bandwidth',
    'MACD_12_26_9_line', 'MACD_12_26_9_signal', 'MACD_12_26_9_histogram',
    'ATR_14', 'ATR_14_pct',
    'ADX_14', 'ADX_14_plus_di', 'ADX_14_minus_di',
    'STOCH_14_3_K', 'STOCH_14_3_D',
    'PSAR', 'PSAR_trend',
    'round_10_above_dist_pips', 'round_10_below_dist_pips', 'round_50_closest_dist_pips',
    # Add features generated by apply_candle_features (vectorized ones)
    # Example names: 'body_size_lag1', 'upper_wick_lag1', etc. Need actual names.
    # Add features from S/R metrics if calculated
    'direccion_ultimo', 'fuerza_ultimo', 'pips_desde_ultimo', 'minutos_desde_ultimo_touch',
    'tendencia_recent', 'pips_cercano', 'minutos_desde_cercano', 'fuerza_nivel_cercano',
    'pips_nivel_arriba', 'pips_nivel_abajo'
]

# Features generated by candle vectorization (example names)
candle_vector_features = [f'{feat}_lag{i}' for feat in ['body_size', 'upper_wick', 'lower_wick', 'total_range', 'body_ratio', 'upper_wick_ratio', 'lower_wick_ratio'] for i in range(1, 7)] # Adjust n_candles

# Combine lists - This needs to be dynamic based on pipeline steps
all_numerical_features = base_numerical_features # + candle_vector_features # Add dynamically

# Categorical features (example, adjust based on actual features)
categorical_features = [
    'sma_order_score', # Example if generated
    'crossover', # Example if generated
    'PSAR_trend_changed', # Example if generated
    'candle_cluster', # If CandleClusterer is used
    'pattern' # If pattern generation is used
    # Add S/R categorical features
    'comportamiento', 'posicion_precio_recent', 'nivel_en_recent', 'posicion_nivel_cercano'
]

# Define the final feature list *after* pipeline definition might be safer
# For now, we proceed with these examples.

# Crear Pipeline de Preprocesamiento y Modelo

Construir un `Pipeline` de Scikit-Learn que incluya:
1.  Generación de la variable objetivo (`generate_target_variable`).
2.  Aplicación de indicadores técnicos (`apply_technical_indicators`).
3.  Cálculo de características de velas y vectorización (`apply_candle_features`).
4.  (Opcional) Clustering de velas (`CandleClusterer`).
5.  Selección de las características finales a usar.
6.  Preprocesamiento de características numéricas (Imputación con `SimpleImputer`, Escalamiento con `StandardScaler`).
7.  Preprocesamiento de características categóricas (Imputación con `SimpleImputer(strategy='most_frequent')`, Codificación con `OneHotEncoder`).
8.  El modelo `RandomForestClassifier` como paso final.

Usaremos `ColumnTransformer` para aplicar diferentes preprocesamientos a columnas numéricas y categóricas.

In [ ]:
# Define the target variable name
TARGET_VARIABLE = 'multiclass_target'

# --- Define Pipeline Steps ---

# 1. Feature Engineering Steps (using FunctionTransformer)
# Note: Target generation should ideally happen *before* splitting data,
# but for pipeline demonstration, we include it. Be mindful of data leakage if
# target depends on future info relative to the split point.
# A safer approach is to generate target first, then split, then run feature engineering pipeline.
# Let's assume target is generated first for this example structure.

pipeline_steps = []

# Step: Apply Technical Indicators
pipeline_steps.append(('tech_indicators', FunctionTransformer(apply_technical_indicators, validate=False)))

# Step: Apply Candle Features
pipeline_steps.append(('candle_features', FunctionTransformer(apply_candle_features, kw_args={'n_candles': 6}, validate=False)))

# Step: (Optional) Apply Candle Clustering
# pipeline_steps.append(('candle_clustering', CandleClusterer(n_clusters=3, random_state=42)))


# --- Define Preprocessing for Numerical and Categorical Features ---

# Create lists of numerical and categorical feature names *dynamically* after engineering steps
# This is tricky within a static pipeline definition. A common approach is to run
# feature engineering once to get column names, then define the preprocessor.
# Or, use custom transformers that handle column selection internally.

# For this example, let's assume we know the final column names after the steps above.
# We'll refine `all_numerical_features` and `categorical_features` (defined earlier)

# Placeholder: Manually list expected features after engineering steps
# This is fragile and needs adjustment based on actual function outputs.
final_numerical_features = [ # RE-DEFINE THIS BASED ON ACTUAL OUTPUTS
    'SMA_10', 'SMA_40', 'SMA_100', 'diff_SMA_10_SMA_40', 'ratio_SMA_10_SMA_40',
    'RSI_14', 'RSI_7', 'RSI_21', 'dist_rsi_70', 'dist_rsi_30', 'RSI_14_slope',
    'BB_20_20_middle', 'BB_20_20_upper', 'BB_20_20_lower', 'BB_20_20_percent_b', 'BB_20_20_bandwidth',
    'MACD_12_26_9_line', 'MACD_12_26_9_signal', 'MACD_12_26_9_histogram', 'MACD_12_26_9_line_slope',
    'ATR_14', 'ATR_14_pct', 'ATR_14_change',
    'ADX_14', 'ADX_14_plus_di', 'ADX_14_minus_di', 'ADX_14_slope',
    'STOCH_14_3_K', 'STOCH_14_3_D', 'STOCH_14_3_K_slope',
    'PSAR', # PSAR value itself
    'round_10_above_dist_pips', 'round_10_below_dist_pips', 'round_50_closest_dist_pips',
    'body_size', 'upper_wick', 'lower_wick', # Current candle features
    # Add vectorized candle features (example names)
    'body_size_lag1', 'body_size_lag2', 'body_size_lag3', 'body_size_lag4', 'body_size_lag5',
    'upper_wick_lag1', 'upper_wick_lag2', 'upper_wick_lag3', 'upper_wick_lag4', 'upper_wick_lag5',
    'lower_wick_lag1', 'lower_wick_lag2', 'lower_wick_lag3', 'lower_wick_lag4', 'lower_wick_lag5',
    # Add S/R numerical features if available and calculated
    'fuerza_ultimo', 'pips_desde_ultimo', 'minutos_desde_ultimo_touch',
    'pips_cercano', 'minutos_desde_cercano', 'fuerza_nivel_cercano',
    'pips_nivel_arriba', 'pips_nivel_abajo'
]

final_categorical_features = [ # RE-DEFINE THIS
    'sma_order_score', 'crossover', 'PSAR_trend', 'PSAR_trend_changed',
    #'candle_cluster', # If clusterer step is included
    #'pattern', # If pattern step is included
    # Add S/R categorical features if available
    'direccion_ultimo', 'comportamiento', 'posicion_precio_recent',
    'tendencia_recent', 'nivel_en_recent', 'posicion_nivel_cercano'
]


# Numerical Preprocessor
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # Handle NaNs from calculations/lags
    ('scaler', StandardScaler()) # Scale numerical features
])

# Categorical Preprocessor
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Handle potential NaNs
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) # One-hot encode
])

# --- Create ColumnTransformer ---
# This transformer applies the correct preprocessing to the correct columns

# Need a function to dynamically get columns for the preprocessor *after* feature engineering
# This is a limitation of static pipeline definition.
# Workaround: Define preprocessor with known columns, ensure feature engineering produces them.

# Let's assume the feature engineering steps reliably produce columns named
# according to `final_numerical_features` and `final_categorical_features`.

# Custom transformer to select columns before preprocessing
class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        print(f"Selecting columns: {self.columns}")
        # Ensure X is a DataFrame
        if not isinstance(X, pd.DataFrame):
             # This might happen if a previous step outputs numpy array
             # Need to handle column names carefully through the pipeline
             raise TypeError("ColumnSelector expects a Pandas DataFrame")
        
        available_cols = [col for col in self.columns if col in X.columns]
        missing = set(self.columns) - set(available_cols)
        if missing:
             print(f"Warning: Columns missing in ColumnSelector: {missing}")
        
        return X[available_cols]


preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, final_numerical_features),
        ('cat', categorical_transformer, final_categorical_features)
    ],
    remainder='drop' # Drop columns not specified in transformers
    # Use 'passthrough' if you want to keep other columns (careful with data types)
)

# Step: Add Preprocessor to pipeline steps
pipeline_steps.append(('preprocessing', preprocessor))


# --- Define Model ---
# Step: Add RandomForestClassifier Model
rf_model = RandomForestClassifier(
    n_estimators=100, # Increased estimators
    max_depth=20,
    min_samples_split=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced', # Good for imbalanced targets
    max_features='sqrt'
)

pipeline_steps.append(('classifier', rf_model))


# --- Create the Final Pipeline ---
full_pipeline = Pipeline(steps=pipeline_steps)

print("Pipeline created successfully:")
print(full_pipeline)

# Dividir Datos en Entrenamiento y Prueba

Dividir el DataFrame en conjuntos de entrenamiento y prueba basados en una fecha específica (por ejemplo, antes y después de '20230101'). Separar características (X) y variable objetivo (y).

*Importante:* La generación de la variable objetivo (`generate_target_variable`) debe hacerse *antes* de la división para asegurar que `y_train` y `y_test` estén definidos.

In [ ]:
# First, generate the target variable on the entire dataset
# We assume df_EURUSD_5min has the necessary OHLC columns
# Ensure 'datetime' column exists and is sorted
df_EURUSD_5min = df_EURUSD_5min.sort_values('datetime').reset_index(drop=True)

# Generate target (this modifies the DataFrame)
df_processed = generate_target_variable(df_EURUSD_5min, pips_objetivo=10, pips_stop=3, future_periods=24)

# Drop rows where target could not be calculated (usually at the end due to future lookahead)
df_processed = df_processed.dropna(subset=[TARGET_VARIABLE, 'compra_resultado', 'venta_resultado']) # Also drop rows missing evaluation results
# Also drop rows where future values were NaN if needed for feature engineering
df_processed = df_processed.dropna(subset=['future_high_values', 'future_low_values', 'future_close_values'], how='any')


# Define the split date
split_date = "2023-01-01" # Use YYYY-MM-DD format for clarity

# Ensure 'datetime' column is suitable for comparison
if 'datetime' not in df_processed.columns:
     # If 'Date' column exists from previous notebook version
     if 'Date' in df_processed.columns:
         df_processed['datetime'] = pd.to_datetime(df_processed['Date'], format='%Y%m%d', errors='coerce') # Adjust format if needed
     else:
          raise KeyError("Missing 'datetime' column for splitting.")

# Convert split_date to the same timezone as the data if necessary
if df_processed['datetime'].dt.tz is not None:
    split_datetime = pd.Timestamp(split_date, tz=df_processed['datetime'].dt.tz)
else:
    split_datetime = pd.Timestamp(split_date)


# Split data
df_train = df_processed[df_processed['datetime'] < split_datetime].copy()
df_test = df_processed[df_processed['datetime'] >= split_datetime].copy()

# Separate features (X) and target (y)
# X includes all columns needed for the pipeline's feature engineering steps
# y is just the target variable column
X_train = df_train.drop(columns=[TARGET_VARIABLE])
y_train = df_train[TARGET_VARIABLE]

X_test = df_test.drop(columns=[TARGET_VARIABLE])
y_test = df_test[TARGET_VARIABLE]

# Clean up large intermediate dataframe if memory is a concern
# del df_processed
# import gc
# gc.collect()

print(f"Data split completed:")
print(f"Training set shape: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Test set shape: X_test {X_test.shape}, y_test {y_test.shape}")
print(f"Training target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest target distribution:\n{y_test.value_counts(normalize=True)}")

# Display first few rows of training features
X_train.head()

# Entrenar el Pipeline (Preprocesamiento + Modelo)

Entrenar el Pipeline completo usando los datos de entrenamiento (`X_train`, `y_train`). El Pipeline aplicará automáticamente todos los pasos de preprocesamiento definidos (indicadores técnicos, características de velas, imputación, escalado, codificación) y luego entrenará el RandomForest.

In [ ]:
print("Starting pipeline training...")
# Train the entire pipeline on the training data
# The pipeline handles all transformations internally
try:
    full_pipeline.fit(X_train, y_train)
    print("Pipeline training completed successfully.")
except Exception as e:
    print(f"An error occurred during pipeline fitting: {e}")
    # Add more detailed error handling or debugging if needed
    # For example, check intermediate steps or data shapes/types

# Note: Fitting might take time depending on data size and complexity of steps.

# Evaluar el Modelo en Entrenamiento y Prueba

Usar el Pipeline entrenado para hacer predicciones en los conjuntos de entrenamiento (`X_train`) y prueba (`X_test`). Calcular y mostrar el `classification_report` y `accuracy_score` para ambos conjuntos. Esto ayuda a verificar el rendimiento y detectar posible overfitting.

In [ ]:
print("Evaluating model performance...")

# Predict on training set
try:
    y_pred_train = full_pipeline.predict(X_train)
    acc_train = accuracy_score(y_train, y_pred_train)
    print("\n=== Reporte de Clasificación - Conjunto de Entrenamiento ===\n")
    print(classification_report(y_train, y_pred_train))
    print(f"Accuracy Train: {acc_train:.4f}\n")
except Exception as e:
    print(f"Error predicting or evaluating on training set: {e}")


# Predict on test set
try:
    y_pred_test = full_pipeline.predict(X_test)
    acc_test = accuracy_score(y_test, y_pred_test)
    print("\n=== Reporte de Clasificación - Conjunto de Prueba ===\n")
    print(classification_report(y_test, y_pred_test))
    print(f"Accuracy Test: {acc_test:.4f}\n")
except Exception as e:
    print(f"Error predicting or evaluating on test set: {e}")

# Compare train and test accuracy to check for overfitting
if 'acc_train' in locals() and 'acc_test' in locals():
    print(f"Train Accuracy: {acc_train:.4f}")
    print(f"Test Accuracy:  {acc_test:.4f}")
    if acc_train > acc_test + 0.1: # Example threshold for overfitting check
        print("Warning: Potential overfitting detected (Train accuracy significantly higher than test accuracy).")

# Analizar Importancia de Características

Extraer la importancia de las características del modelo RandomForest dentro del Pipeline. Visualizar las características más importantes.

*Nota:* Obtener los nombres correctos de las características después del `OneHotEncoder` requiere acceder al transformador dentro del `ColumnTransformer` del pipeline.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

try:
    # Access the preprocessing step and then the categorical transformer's OneHotEncoder
    preprocessor_step = full_pipeline.named_steps['preprocessing']
    categorical_transformer_step = preprocessor_step.named_transformers_['cat']
    onehot_encoder = categorical_transformer_step.named_steps['onehot']

    # Get feature names after one-hot encoding
    onehot_feature_names = onehot_encoder.get_feature_names_out(final_categorical_features) # Use the list passed to the transformer

    # Get numerical feature names (assuming they pass through unchanged in name by scaler/imputer)
    # The order should match the order in the ColumnTransformer definition
    numerical_feature_names = final_numerical_features

    # Combine all feature names in the correct order
    all_feature_names = list(numerical_feature_names) + list(onehot_feature_names)

    # Access the classifier step and get feature importances
    classifier_step = full_pipeline.named_steps['classifier']
    importances = classifier_step.feature_importances_

    # Create DataFrame for visualization
    importance_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)

    # Select top N features
    top_n = 40
    top_features = importance_df.head(top_n)

    # Plot
    plt.figure(figsize=(10, max(10, top_n // 2))) # Adjust height based on N
    ax = sns.barplot(x='Importance', y='Feature', data=top_features, palette='viridis')
    plt.title(f'Top {top_n} Most Important Features (Random Forest from Pipeline)')
    plt.xlabel('Importance Score')
    plt.ylabel('Feature Name')
    plt.tight_layout()

    # Add values to the bars
    for i, v in enumerate(top_features['Importance']):
         # Adjust text position slightly for better readability
         ax.text(v + 0.0005, i, f'{v:.4f}', va='center', fontsize=9)

    plt.show()

    # Print the numeric values
    print(f"\nTop {top_n} features by importance:")
    print(top_features.to_string())

except Exception as e:
    print(f"Could not extract or plot feature importances: {e}")
    # This might happen if pipeline structure is different or steps failed

# Analizar Métricas por Umbral de Probabilidad

Obtener las probabilidades predichas para el conjunto de prueba (`predict_proba`). Usar la función `analizar_metricas_por_umbral` para evaluar el rendimiento (profit, precisión, pips promedio) en diferentes umbrales para decisiones de compra y venta. Esto ayuda a determinar umbrales óptimos para generar señales de trading basadas en la confianza del modelo.

In [ ]:
print("Analyzing performance by probability threshold...")

try:
    # Get predicted probabilities for the test set
    # Output shape: (n_samples, n_classes)
    y_pred_proba_test = full_pipeline.predict_proba(X_test)

    # Get the class labels in the order the model uses them
    class_labels = full_pipeline.classes_ # Or full_pipeline.named_steps['classifier'].classes_

    # Ensure the original df_test has the necessary columns for analysis
    # ('compra_resultado', 'venta_resultado', 'compra_pips', 'venta_pips', 'datetime')
    # We should use the df_test *before* dropping the target, or merge results back.
    # Let's use the df_test DataFrame created during the split, which contains these columns.

    # Check if df_test contains the required columns from target generation
    required_analysis_cols = ['compra_resultado', 'venta_resultado', 'compra_pips', 'venta_pips', 'datetime']
    if not all(col in df_test.columns for col in required_analysis_cols):
        print("Warning: df_test is missing columns required for threshold analysis. Merging back...")
        # If X_test was used, it doesn't have these. Need the original df_test slice.
        # Recreate df_test slice if needed (assuming df_processed is available)
        # df_test_analysis = df_processed[df_processed['datetime'] >= split_datetime].copy()
        # Or merge based on index if indices are aligned
        df_test_analysis = df_test.copy() # Use the one from split which should have the columns
        # Add predictions and probabilities back if needed, ensuring index alignment
        df_test_analysis['prediction'] = y_pred_test
        # Add probabilities for each class
        for i, label in enumerate(class_labels):
             df_test_analysis[f'proba_{label}'] = y_pred_proba_test[:, i]

    else:
        df_test_analysis = df_test.copy() # Already has the columns
        # Add predictions and probabilities
        df_test_analysis['prediction'] = y_pred_test
        for i, label in enumerate(class_labels):
             df_test_analysis[f'proba_{label}'] = y_pred_proba_test[:, i]


    # Run the analysis function
    # Ensure the function knows which probability columns correspond to buy/sell targets
    # Assuming 'objetivo_compra_alcanzado' is index 0 and 'objetivo_venta_alcanzado' is index 2 (adjust if needed)
    print(f"Model classes: {class_labels}") # Verify order
    # Find index for buy/sell targets
    try:
        buy_target_index = list(class_labels).index('objetivo_compra_alcanzado')
        sell_target_index = list(class_labels).index('objetivo_venta_alcanzado')
        print(f"Buy target index: {buy_target_index}, Sell target index: {sell_target_index}")
    except ValueError:
        print("Error: Could not find standard buy/sell target names in model classes.")
        buy_target_index, sell_target_index = -1, -1 # Indicate error


    if buy_target_index != -1 and sell_target_index != -1:
        compra_df, venta_df, compra_mensual_df, venta_mensual_df = analizar_metricas_por_umbral(
            y_pred_proba=y_pred_proba_test,
            df_test_results=df_test_analysis, # Pass the DataFrame with results and probabilities
            pip_objetivo=10, # Must match target generation
            pip_stop=3,      # Must match target generation
            compra_proba_col_index=buy_target_index, # Index for 'objetivo_compra_alcanzado' probability
            venta_proba_col_index=sell_target_index  # Index for 'objetivo_venta_alcanzado' probability
        )

        # --- Display Results ---
        print("\n--- Threshold Analysis Results ---")

        # Find and print best threshold for buys based on profit
        if not compra_df.empty:
            mejor_umbral_compra = compra_df.loc[compra_df['profit'].idxmax()]
            print(f"\nBest Threshold for Buys (Max Profit):")
            print(f"  Threshold: {mejor_umbral_compra['threshold']:.2f}")
            print(f"  Profit: {mejor_umbral_compra['profit']:,.0f} pips")
            print(f"  Precision: {mejor_umbral_compra['precision']:.2%}")
            print(f"  Avg Pips/Trade: {mejor_umbral_compra['avg_pips']:.2f}")
            print(f"  Total Trades: {mejor_umbral_compra['total_trades']}")

            # Display monthly breakdown for the best buy threshold
            umbral_optimo_compra = mejor_umbral_compra['threshold']
            analisis_mensual_compra = compra_mensual_df[compra_mensual_df['threshold'] == umbral_optimo_compra]
            print(f"\nMonthly Analysis for Buys (Threshold {umbral_optimo_compra:.2f}):")
            print(analisis_mensual_compra.sort_values('month')[['month', 'precision', 'total_trades', 'profit', 'avg_pips']].round(2))
        else:
            print("\nNo buy trades found for analysis.")


        # Find and print best threshold for sells based on profit
        if not venta_df.empty:
            mejor_umbral_venta = venta_df.loc[venta_df['profit'].idxmax()]
            print(f"\nBest Threshold for Sells (Max Profit):")
            print(f"  Threshold: {mejor_umbral_venta['threshold']:.2f}")
            print(f"  Profit: {mejor_umbral_venta['profit']:,.0f} pips")
            print(f"  Precision: {mejor_umbral_venta['precision']:.2%}")
            print(f"  Avg Pips/Trade: {mejor_umbral_venta['avg_pips']:.2f}")
            print(f"  Total Trades: {mejor_umbral_venta['total_trades']}")

            # Display monthly breakdown for the best sell threshold
            umbral_optimo_venta = mejor_umbral_venta['threshold']
            analisis_mensual_venta = venta_mensual_df[venta_mensual_df['threshold'] == umbral_optimo_venta]
            print(f"\nMonthly Analysis for Sells (Threshold {umbral_optimo_venta:.2f}):")
            print(analisis_mensual_venta.sort_values('month')[['month', 'precision', 'total_trades', 'profit', 'avg_pips']].round(2))
        else:
             print("\nNo sell trades found for analysis.")

    else:
        print("Skipping threshold analysis due to missing target class indices.")


except Exception as e:
    print(f"An error occurred during threshold analysis: {e}")
    import traceback
    traceback.print_exc()

# Guardar el Modelo Entrenado

Guardar el objeto Pipeline entrenado completo en un archivo usando `joblib.dump`. Esto permite recargar el pipeline más tarde para hacer predicciones sobre nuevos datos sin necesidad de reentrenar.

In [ ]:
# Define file path for saving the model
model_dir = '../models' # Create this directory if it doesn't exist
os.makedirs(model_dir, exist_ok=True)
model_filename = 'forex_rf_pipeline_eurusd_5min.joblib'
model_filepath = os.path.join(model_dir, model_filename)

try:
    # Save the entire pipeline object
    joblib.dump(full_pipeline, model_filepath)
    print(f"Pipeline saved successfully to: {model_filepath}")

    # Optional: Save associated information like feature lists or class labels
    metadata = {
        'description': 'Full Scikit-Learn pipeline for EURUSD 5min Forex prediction (RF). Includes feature engineering and preprocessing.',
        'target_variable': TARGET_VARIABLE,
        'model_step_name': 'classifier',
        'class_labels': list(full_pipeline.classes_),
        'split_date': split_date,
        # Add feature lists if needed, but they are implicitly handled by the pipeline
        # 'numerical_features': final_numerical_features,
        # 'categorical_features': final_categorical_features
    }
    metadata_filepath = os.path.join(model_dir, 'forex_rf_pipeline_eurusd_5min_metadata.json')
    with open(metadata_filepath, 'w') as f:
        json.dump(metadata, f, indent=4)
    print(f"Metadata saved successfully to: {metadata_filepath}")


except Exception as e:
    print(f"Error saving the pipeline or metadata: {e}")

# Example of how to load the pipeline later:
# loaded_pipeline = joblib.load(model_filepath)
# print("Pipeline loaded successfully.")
# new_predictions = loaded_pipeline.predict(new_data)